In [19]:
from collections import defaultdict

def read_run(path):
    run = defaultdict(dict)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            qid, _, docid, rank, score, system = line.strip().split()

            run[qid][docid] = {
                "rank": int(rank),
                "score": float(score)
            }

    return run

bm25 = read_run("runs/bm25.trec")
knn = read_run("runs/knn.trec")

print("Queries BM25:", len(bm25))
print("Queries KNN :", len(knn))

Queries BM25: 10
Queries KNN : 10


In [20]:
def rrf_score(rank, k=60):
    return 1 / (k + rank)

In [21]:
hybrid = {}

for qid in bm25:

    docs = {}

    for docid, info in bm25[qid].items():
        docs.setdefault(docid, 0)
        docs[docid] += rrf_score(info["rank"])

    for docid, info in knn[qid].items():
        docs.setdefault(docid, 0)
        docs[docid] += rrf_score(info["rank"])

    ranking = sorted(
        docs.items(),
        key=lambda x: x[1],
        reverse=True
    )

    hybrid[qid] = ranking

print("Ranking híbrido criado.")

Ranking híbrido criado.


In [22]:
with open("runs/hybrid.trec", "w", encoding="utf-8") as f:
    
    for qid, ranking in hybrid.items():

        for rank, (docid, score) in enumerate(ranking[:100], 1):

            f.write(
                f"{qid} Q0 {docid} "
                f"{rank} {score:.6f} hybrid\n"
            )

print("Arquivo salvo: hybrid.trec")

Arquivo salvo: hybrid.trec
